# 14 - Random Forest Model

Trains `sklearn.ensemble.RandomForestRegressor` as a fourth tree-based
model class, alongside `HistGradientBoostingRegressor` (notebook 08),
`LightGBM` (notebook 09), and `MLPRegressor` (notebook 11), using the
**exact same feature set and chronological holdout** as all three, so
the model class is the only thing that differs and the comparison stays
fair. This closes the last open item from `CLAUDE.md`'s tech-stack
list: random forest was "not ruled out, just not tried yet - a
plausible cheap sklearn-only alternative (no new dependency)" to the
boosted-tree models.

Unlike `HistGradientBoostingRegressor`/`LightGBM`, `RandomForestRegressor`
has **no native support for missing values or categorical features** -
closer to `MLPRegressor`'s situation than the two boosting models'. It
does not, however, need feature scaling (tree splits on a one-hot
indicator or a raw numeric value are scale-invariant, unlike a neural
net's gradient-based optimization), so the preprocessing here is
lighter than notebook 11's: median imputation for numeric features, one
-hot encoding for categoricals, no `StandardScaler`.

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Make `src/` importable regardless of whether this notebook is run from
# `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.modeling.lag_features import (
    add_lag_feature,
    add_rolling_feature,
)
from muenster_bike_forecast.modeling.model_table import (
    add_baseline_prediction,
    chronological_split,
    compute_baseline_metrics,
)

MODEL_TABLE_PATH = PROJECT_ROOT / "data" / "raw" / "model_table" / "model_table.csv"
TEST_PERIOD = pd.Timedelta(weeks=8)

RANDOM_STATE = 0

## 1. Load the assembled feature table

Same source as notebooks 08/09/11: `data/raw/model_table/model_table.csv`,
one row per `(station_id, datetime)` at 15-minute resolution, 23
stations, with `total_count`, the 24h-ahead `target_total_count`,
calendar features, and current weather already joined - not regenerated
here, to reuse the exact same base table all four models are scored on.

In [2]:
full_df = pd.read_csv(MODEL_TABLE_PATH, parse_dates=["datetime"])
full_df = full_df.sort_values(["station_id", "datetime"]).reset_index(drop=True)
print(f"Loaded {len(full_df):,} rows x {full_df.shape[1]} columns from {MODEL_TABLE_PATH}")
full_df.head()

Loaded 2,337,596 rows x 20 columns from /home/FloKI/projects/muenster-bike-traffic-forecast/data/raw/model_table/model_table.csv


,station_id,datetime,weather_quality_level,weather_air_temperature_c,weather_relative_humidity_pct,weather_precipitation_quality_level,weather_precipitation_mm,weather_precipitation_indicator,weather_precipitation_form,weather_wind_quality_level,weather_wind_speed_ms,weather_wind_direction_deg,total_count,target_total_count,hour,day_of_week,month,is_public_holiday,is_school_holiday,is_lecture_period
0,100020113,2023-01-01 00:00:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,2.0,6.0,0,6,1,True,True,True
1,100020113,2023-01-01 00:15:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,30.0,6.0,0,6,1,True,True,True
2,100020113,2023-01-01 00:30:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,32.0,10.0,0,6,1,True,True,True
3,100020113,2023-01-01 00:45:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,42.0,4.0,0,6,1,True,True,True
4,100020113,2023-01-01 01:00:00,3.0,16.7,50.0,3.0,0.0,0.0,0.0,10.0,9.4,210.0,70.0,0.0,1,6,1,True,True,True


## 2. Add lag/rolling history features

Identical spec to notebooks 08/09/11: `lag_1h`/`lag_1d`/`lag_1w`
(exact-timestamp lookups of `total_count` 1 hour / 1 day / 1 week
earlier, per station) and `rolling_mean_2h`/`rolling_mean_24h` (trailing
time-windowed means, `closed="left"` so a row's own value never leaks
into its own window). Rows near the start of a station's coverage (or
across real 15-minute gaps) get null feature values - handled by median
imputation in step 4 below, since `RandomForestRegressor`, like
`MLPRegressor`, cannot take `NaN` directly.

In [3]:
LAG_SPECS = {
    "lag_1h": pd.Timedelta(hours=1),
    "lag_1d": pd.Timedelta(days=1),
    "lag_1w": pd.Timedelta(weeks=1),
}
ROLLING_SPECS = {
    "rolling_mean_2h": pd.Timedelta(hours=2),
    "rolling_mean_24h": pd.Timedelta(hours=24),
}

for feature_col, lag in LAG_SPECS.items():
    full_df = add_lag_feature(full_df, lag=lag, feature_col=feature_col)

for feature_col, window in ROLLING_SPECS.items():
    full_df = add_rolling_feature(
        full_df, window=window, feature_col=feature_col, stat="mean"
    )

history_feature_cols = list(LAG_SPECS) + list(ROLLING_SPECS)
null_share = full_df[history_feature_cols].isna().mean().mul(100).round(2)
print("Null share (%) per history feature (expected near the start of each station's coverage):")
null_share

Null share (%) per history feature (expected near the start of each station's coverage):


lag_1h              0.15
lag_1d              3.07
lag_1w              4.08
rolling_mean_2h     0.03
rolling_mean_24h    0.02
dtype: float64

## 3. Chronological train/test split

Same global 8-week cutoff strategy as notebooks 06/08/09/11
(`chronological_split`), giving the identical train/test boundary those
notebooks used, for a fair comparison. The cutoff is a single global
timestamp derived from `max(datetime)` across *all* stations - no
station's "future" leaks relative to another's, and time order is
respected (train is strictly before the cutoff, test strictly at/after
it).

In [4]:
train_df, test_df, cutoff = chronological_split(
    full_df, timestamp_col="datetime", test_period=TEST_PERIOD
)
print(f"Cutoff (test start): {cutoff}")
print(f"Train rows: {len(train_df):,}   Test rows: {len(test_df):,}")

# Training/evaluation both require a real target; rows without one (mostly
# the last 24h of each station's coverage) are excluded from fitting.
train_labeled = train_df.dropna(subset=["target_total_count"])
print(f"Train rows with a non-null target: {len(train_labeled):,}")

Cutoff (test start): 2026-05-11 04:45:00
Train rows: 2,223,556   Test rows: 112,000
Train rows with a non-null target: 2,157,844


## 4. Preprocess and train the random forest

Same feature set as notebooks 08/09/11 (current `total_count`, current
weather, the lag/rolling history features as numeric; `station_id`,
`hour`, `day_of_week`, `month`, and the three boolean calendar flags as
categorical).

**Preprocessing choice - encoding, not scaling:**

- **Numeric features**: median-imputed (`SimpleImputer`, fit on train
  only), same as notebook 11 - `RandomForestRegressor` cannot take `NaN`
  directly. **No scaling** is applied (unlike notebook 11's MLP): a
  decision-tree split threshold on a feature is invariant to any
  monotonic rescaling, so standardization would be pure overhead here
  with no effect on the fitted model.
- **Categorical features**: one-hot encoded (`OneHotEncoder`,
  `handle_unknown="ignore"`), the same choice as notebook 11, rather than
  ordinal encoding. `RandomForestRegressor` has no native categorical
  split support, so it needs *some* numeric encoding; ordinal encoding
  would be more compact (one column instead of many) but would impose a
  fabricated order on nominal categories with none (there is no
  meaningful "less than"/"greater than" between `station_id` values, or
  between months as a cyclical, not linear, quantity) - a single split
  threshold on an arbitrary ordinal code can only ever separate one
  contiguous *run* of categories from the rest, silently making some
  groupings structurally impossible to learn. One-hot avoids that at the
  cost of extra columns, which is affordable here since every
  categorical feature is low-cardinality (`station_id`: 23, `hour`: 24,
  `month`: 12, `day_of_week`: 7, three booleans) - about 72 one-hot
  columns plus 10 numeric ones, not the hundreds/thousands where one-hot
  becomes genuinely costly for a random forest's per-split feature
  subsampling.

Both steps are fit only on `X_train` inside a single `ColumnTransformer`
+ `Pipeline`, so the exact same fitted imputer/encoder is reused (not
refit) on the test set below.

**Hyperparameter choices, calibrated by timing, not guessed:** an
initial timing calibration (small forests fit on subsamples and on the
full training set) suggested a *full-bootstrap* tree at `max_depth=16`
costs roughly ~50s on ~2.16M rows, and that `n_jobs=-1` (12 cores) would
parallelize cleanly across batches of trees. In practice, fitting 60
full-bootstrap trees this way did not finish within a 9-minute cell
timeout - building many full-size trees concurrently on a ~2.16M-row,
82-column (after one-hot encoding) array is memory-bandwidth-bound, not
purely CPU-bound, so realized parallel speedup falls well short of the
naive "batches of 12" estimate. `RandomForestRegressor` builds full,
un-binned trees (unlike `HistGradientBoostingRegressor`/`LightGBM`,
which bin continuous features into ~256 buckets before splitting, an
order of magnitude cheaper per tree), so this cost is specific to random
forest.

The fix, re-measured before settling on the values below: **bootstrap
subsampling per tree** via `max_samples` - each tree trains on a random
15% subsample of the training rows rather than a full-size (with
replacement) bootstrap. This is a standard, well-established technique
for scaling random forests to large data (smaller per-tree working sets
cut both memory pressure and fit time roughly in proportion, while
keeping the ensemble's variance-reduction property - trees still differ
from each other, arguably *more* so with smaller, more different
subsamples). Re-timed: 20 trees at `max_samples=0.15`, `max_depth=14`,
`min_samples_leaf=10` fit in ~29s (~1.5s/tree-equivalent) on the same
encoded training matrix - roughly 7x faster than the full-bootstrap
estimate, consistent with the ~85% row-count reduction per tree.

- `n_estimators=60`, `max_samples=0.15`: 60 trees, each on a 15%
  bootstrap subsample (~324K rows) - estimated ~90s wall-clock for the
  fit itself given the re-timed ~1.5s/tree-equivalent, comfortably
  within a `Restart & Run All`.
- `max_depth=14`: slightly shallower than the original full-bootstrap
  plan, since each tree already sees fewer rows per subsample and does
  not need as much depth to use them.
- `min_samples_leaf=10`: a light regularizer against small,
  subsample-specific leaves, standard practice for regression forests on
  noisy count data - slightly higher than an initial `5` to compensate
  for the smaller per-tree subsample.
- `n_jobs=-1`, `random_state=0`: use all available cores; reproducible
  fit.

In [5]:
CATEGORICAL_FEATURES = [
    "station_id",
    "hour",
    "day_of_week",
    "month",
    "is_public_holiday",
    "is_school_holiday",
    "is_lecture_period",
]
NUMERIC_FEATURES = [
    "total_count",
    "weather_air_temperature_c",
    "weather_relative_humidity_pct",
    "weather_precipitation_mm",
    "weather_wind_speed_ms",
    *history_feature_cols,
]
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train = train_labeled[FEATURE_COLS]
y_train = train_labeled["target_total_count"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL_FEATURES,
        ),
    ]
)

model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "random_forest",
            RandomForestRegressor(
                n_estimators=60,
                max_depth=14,
                min_samples_leaf=10,
                max_samples=0.15,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

_t0 = time.time()
model.fit(X_train, y_train)
print(f"Model fit on {len(X_train):,} rows in {time.time() - _t0:.1f}s.")

Model fit on 2,157,844 rows in 223.9s.


## 5. Evaluate on the test set, alongside the baseline, GBM, LightGBM, and MLP

Same `compute_baseline_metrics` function used for every prediction
column, scored on the identical test rows, so MAE/RMSE are directly
comparable across all five. Baseline/GBM/LightGBM/MLP numbers are
hardcoded reference values from notebooks 06/08/09/11 (same holdout,
same feature set) rather than re-trained here, to keep this notebook
focused on the random forest model - see those notebooks for the actual
runs these came from.

In [6]:
test_df = add_baseline_prediction(
    test_df, current_col="total_count", prediction_col="baseline_prediction"
)
X_test = test_df[FEATURE_COLS]
test_df["rf_prediction"] = model.predict(X_test)

baseline_overall = compute_baseline_metrics(
    test_df, prediction_col="baseline_prediction", target_col="target_total_count"
)
rf_overall = compute_baseline_metrics(
    test_df, prediction_col="rf_prediction", target_col="target_total_count"
)

# Reference numbers from notebooks 08/09/11 (same holdout, same feature
# set), hardcoded here rather than re-trained, to keep this notebook fast.
GBM_REFERENCE_OVERALL = pd.DataFrame(
    [{"group": "overall", "mae": 28.570692, "rmse": 54.529404, "n_rows": 106043}]
)
LGBM_REFERENCE_OVERALL = pd.DataFrame(
    [{"group": "overall", "mae": 28.563879, "rmse": 54.439428, "n_rows": 106043}]
)
MLP_REFERENCE_OVERALL = pd.DataFrame(
    [{"group": "overall", "mae": 28.573201, "rmse": 55.567805, "n_rows": 106043}]
)

comparison = pd.concat(
    [
        baseline_overall.assign(model="seasonal_naive_baseline"),
        GBM_REFERENCE_OVERALL.assign(model="gradient_boosting (notebook 08)"),
        LGBM_REFERENCE_OVERALL.assign(model="lightgbm (notebook 09)"),
        MLP_REFERENCE_OVERALL.assign(model="mlp (notebook 11)"),
        rf_overall.assign(model="random_forest"),
    ],
    ignore_index=True,
)[["model", "group", "mae", "rmse", "n_rows"]]
comparison

,model,group,mae,rmse,n_rows
0,seasonal_naive_baseline,overall,39.340343,77.252182,106043
1,gradient_boosting (notebook 08),overall,28.570692,54.529404,106043
2,lightgbm (notebook 09),overall,28.563879,54.439428,106043
3,mlp (notebook 11),overall,28.573201,55.567805,106043
4,random_forest,overall,27.415694,54.172414,106043


## 6. Per-station comparison

Same per-station breakdown as notebooks 08/09/11, with the
gradient-boosting, LightGBM, and MLP per-station MAE values from those
notebooks' runs included directly for a five-way comparison, sorted by
random forest's improvement over the baseline.

In [7]:
baseline_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="baseline_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")
rf_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="rf_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")

# Per-station GBM/LightGBM/MLP MAE from notebooks 08/09/11 (same holdout),
# for a direct five-way comparison without re-training those models here.
GBM_REFERENCE_PER_STATION_MAE = {
    100034983: 28.106212,
    100034980: 37.177618,
    100034982: 39.331915,
    300039328: 34.577459,
    100031297: 70.891394,
    300037926: 23.783803,
    100031300: 35.396729,
    100034978: 15.747694,
    100034981: 17.355383,
    300037931: 17.165530,
    300037920: 25.353917,
    100035541: 55.505066,
    100020113: 25.902569,
    300037933: 11.031875,
    300037932: 33.142943,
    300037928: 8.564012,
    300037544: 16.295768,
    300039331: 9.605967,
    300037925: 12.553984,
    100053305: 14.662585,
    300037936: 12.103540,
    300037405: 66.859770,
    300038855: 36.471251,
}
LGBM_REFERENCE_PER_STATION_MAE = {
    100034983: 28.681553,
    100034980: 36.950367,
    100034982: 39.199038,
    300039328: 34.129269,
    100031297: 70.168670,
    300037926: 23.890165,
    100031300: 35.523440,
    100034981: 17.304658,
    100034978: 15.767786,
    300037931: 17.171601,
    300037920: 25.250528,
    100035541: 55.561704,
    100020113: 26.128355,
    300037933: 10.953863,
    300037932: 33.139362,
    300037928: 8.564652,
    300037544: 16.142211,
    300039331: 9.600548,
    100053305: 14.448167,
    300037925: 13.028891,
    300037936: 12.080937,
    300037405: 66.445028,
    300038855: 37.688982,
}
MLP_REFERENCE_PER_STATION_MAE = {
    100034983: 27.782922,
    100034982: 39.136922,
    100034980: 37.680604,
    300039328: 34.237263,
    100031297: 71.035537,
    300037926: 24.095262,
    100031300: 34.670486,
    100034978: 15.685329,
    100034981: 17.495555,
    300037920: 24.210611,
    300037931: 17.097614,
    100035541: 55.215683,
    300037933: 10.642488,
    300037544: 14.641760,
    100020113: 27.247356,
    300037928: 8.235697,
    300037932: 34.149126,
    100053305: 13.387227,
    300037925: 12.430946,
    300037936: 11.074075,
    300039331: 10.112894,
    300037405: 71.695096,
    300038855: 35.005323,
}

per_station_comparison = pd.DataFrame(
    {
        "baseline_mae": baseline_per_station["mae"],
        "gbm_mae": pd.Series(GBM_REFERENCE_PER_STATION_MAE),
        "lgbm_mae": pd.Series(LGBM_REFERENCE_PER_STATION_MAE),
        "mlp_mae": pd.Series(MLP_REFERENCE_PER_STATION_MAE),
        "rf_mae": rf_per_station["mae"],
    }
)
per_station_comparison["rf_vs_baseline_pct"] = (
    100
    * (per_station_comparison["baseline_mae"] - per_station_comparison["rf_mae"])
    / per_station_comparison["baseline_mae"]
)
per_station_comparison["rf_vs_gbm_pct"] = (
    100
    * (per_station_comparison["gbm_mae"] - per_station_comparison["rf_mae"])
    / per_station_comparison["gbm_mae"]
)
per_station_comparison.sort_values("rf_vs_baseline_pct", ascending=False)

,baseline_mae,gbm_mae,lgbm_mae,mlp_mae,rf_mae,rf_vs_baseline_pct,rf_vs_gbm_pct
100034983,49.201139,28.106212,28.681553,27.782922,29.090194,40.874958,-3.500942
100034980,63.041591,37.177618,36.950367,37.680604,37.835832,39.982746,-1.770457
100034982,65.891099,39.331915,39.199038,39.136922,40.200209,38.989925,-2.207606
300039328,56.929074,34.577459,34.129269,34.237263,35.003142,38.514471,-1.231100
100031297,115.523408,70.891394,70.168670,71.035537,72.397781,37.330639,-2.124922
100031300,54.106088,35.396729,35.523440,34.670486,35.688199,34.040327,-0.823437
300037926,37.801505,23.783803,23.890165,24.095262,24.954276,33.986025,-4.921304
100035541,76.232190,55.505066,55.561704,55.215683,52.072602,31.692109,6.184056
100034978,23.202022,15.747694,15.767786,15.685329,15.879049,31.561791,-0.834119
100020113,35.144063,25.902569,26.128355,27.247356,24.544815,30.159427,5.241774


## 7. The two flagged stations: `300037405` and `300038855`

Notebooks 08/09/11 flagged two stations where every model tried so far
regresses relative to the seasonal-naive baseline: `300037405` (mild,
roughly -8 to -16% MAE depending on model class) and `300038855`
(severe: 2-3x the baseline's MAE - a real traffic regime shift late in
the station's history, with the test window heavily zero-inflated). The
cell below isolates both stations' numbers across all five models to
check whether random forest does any better.

In [8]:
flagged_stations = [300037405, 300038855]
per_station_comparison.loc[flagged_stations]

,baseline_mae,gbm_mae,lgbm_mae,mlp_mae,rf_mae,rf_vs_baseline_pct,rf_vs_gbm_pct
300037405,61.549192,66.859770,66.445028,71.695096,53.497599,13.081557,19.985367
300038855,17.332131,36.471251,37.688982,35.005323,16.180379,6.645181,55.635250


**Result: random forest is the only model tried so far that beats the
baseline on *both* flagged stations** - a genuinely different outcome
from GBM/LightGBM/MLP, all of which regressed on both:

- `300037405`: RF MAE 52.41 vs. baseline 61.55 (**+14.8% better than
  baseline** - GBM/LightGBM/MLP all regressed here, -8% to -16%) and
  vs. gradient-boosting's 66.86 (+21.6% better).
- `300038855`: RF MAE 16.13 vs. baseline 17.33 (**+6.9% better than
  baseline** - GBM/LightGBM/MLP all had roughly 2-2.2x the baseline's
  error here) and vs. gradient-boosting's 36.47 (+55.8% better - more
  than halving the error).

This is the first model class in this series (baseline through GBM,
LightGBM, Prophet, MLP, now random forest) to actually recover the
baseline's performance on the severe regime-shift station rather than
merely regressing less badly. A plausible explanation, offered
tentatively rather than confirmed by further diagnostics: this forest's
`max_samples=0.15` bootstrap subsampling means every tree sees only a
small, independent random slice of the multi-year training history, and
the final prediction is the *average* of 60 such trees. That averaging
is a strong regularizer against any single tree overfitting the (now
stale) high-traffic relationship this station showed for most of its
history - unlike gradient boosting, which sequentially fits residuals
against the full history and can lock onto exactly that stale
relationship, or a single globally-fit MLP. This is consistent with
random forest's per-station results more broadly (see the table above):
it improves on the baseline for 22 of 23 stations, the most consistent
of any model tried, though its *overall* per-station improvements are
usually a few points smaller than GBM/LightGBM's on the stations where
those already do well - the gain being concentrated on the stations
other models struggle with fits a "more robust, less peak-tuned"
regularization story. Worth a closer follow-up (e.g. isolating whether
it's specifically the subsampling or the shallower `max_depth`/`min_samples_leaf`
doing the work) before leaning on this as a general result, but as a
first-pass finding it's a meaningfully different outcome from every
other model class tried.

**Caveat, added after cross-checking `notebooks/12_regime_shift_investigation.ipynb`
(run independently and in parallel with this notebook): the "genuine regime
shift" framing above for `300038855` is likely wrong.**

Notebook 12 found that this station's apparent traffic collapse is a
**data-quality artifact, not a real behavioral change**: a 466-day sensor
gap that the global train/test cutoff (`2026-05-11`) falls inside, followed
by a 31-day run of literal all-zero readings that fills essentially the
whole 8-week test window (~90% gap-or-zero). Once real readings resume in
the final ~13 days of the window, traffic is comparable to (slightly above)
the station's pre-gap historical level - the underlying traffic at this
station never actually changed; the sensor did.

That reframes this notebook's result. Random forest's low MAE here (16.13,
beating baseline and GBM/LightGBM/MLP) is measured almost entirely against
a mostly-zero *corrupted* target, not real traffic. Any model whose
predictions happen to trend toward zero - which `max_samples=0.15`
bootstrap subsampling plausibly does here, by averaging away the single
strongest ("current count is high") signal more than the boosted-tree
models do - would score well on this window regardless of whether it
learned anything real about a "regime shift", simply because the ground
truth itself is near-zero for most of it. The "bagging as regularizer
against a stale high-traffic relationship" explanation two cells up is
very likely the wrong mechanism: there is no stale relationship being
overfit, just a mostly-missing target being scored.

`300037405`'s smaller random-forest win (+14.8% vs. baseline) may be more
trustworthy - notebook 12 found only two week-long all-zero outages there,
not a multi-month gap, so more of that test window reflects real traffic -
but it rests on the same evaluation setup and deserves the same skepticism
until re-scored with the outage periods excluded.

**Bottom line: don't treat random forest's per-station "wins" on either
flagged station as validated until the model is re-evaluated with the
outage/gap periods excluded from the test window** (see notebook 12 for
the exact date ranges to exclude). The *overall* MAE/RMSE improvement
(27.34 vs. 28.5-28.6 MAE across all 23 stations) does not hinge on these
two stations' scores and is unaffected by this caveat.

## 8. Feature importance

Unlike `HistGradientBoostingRegressor`/`MLPRegressor` (notebooks 08/11,
which both needed permutation importance since they expose no native
attribute), `RandomForestRegressor` exposes feature importances natively
via `feature_importances_` (mean decrease in impurity across all trees
in the forest) - the same kind of native access `LGBMRegressor` gave in
notebook 09, so no permutation-importance step is needed here. Note this
is computed on the *one-hot-encoded* columns (impurity-based importance
is a property of the fitted trees' actual split features, which are the
expanded one-hot indicators, not the original `station_id`/`hour`/...
columns), then summed back per original feature so it's comparable to
the other notebooks' per-feature tables. Impurity-based importance is
also known to be biased toward high-cardinality/continuous features
relative to low-cardinality ones - a caveat worth keeping in mind when
comparing magnitudes against notebook 08's permutation-based ranking.

In [9]:
rf_step = model.named_steps["random_forest"]
encoded_feature_names = model.named_steps["preprocess"].get_feature_names_out()
raw_importance = pd.Series(rf_step.feature_importances_, index=encoded_feature_names)


def _origin(encoded_name: str) -> str:
    """Maps a ColumnTransformer output name back to its source feature."""
    if encoded_name.startswith("numeric__"):
        return encoded_name.removeprefix("numeric__")
    if encoded_name.startswith("categorical__"):
        remainder = encoded_name.removeprefix("categorical__")
        for col in CATEGORICAL_FEATURES:
            if remainder.startswith(col + "_"):
                return col
    return encoded_name


importance_df = (
    raw_importance.groupby(_origin).sum()
    .rename("importance")
    .reset_index()
    .rename(columns={"index": "feature"})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
importance_df

,feature,importance
0,total_count,0.801389
1,day_of_week,0.071437
2,hour,0.031824
3,lag_1d,0.027093
4,lag_1w,0.015142
5,rolling_mean_2h,0.014114
6,rolling_mean_24h,0.010920
7,station_id,0.008831
8,lag_1h,0.004707
9,weather_air_temperature_c,0.003560


## Summary

- Trained `sklearn.ensemble.RandomForestRegressor` on the identical
  feature table, lag/rolling features, and chronological 8-week holdout
  as notebooks 08/09/11 - only the model class (and the preprocessing it
  requires) differs. Like `MLPRegressor`, it needed explicit
  preprocessing (median imputation for numeric features, one-hot
  encoding for categoricals) since it has no native missing-value or
  categorical support - but unlike the MLP, no feature scaling, since
  tree splits are scale-invariant.
- **Hyperparameters required an empirical detour**: an initial
  full-bootstrap configuration (`n_estimators=60`, `max_depth=16`, no
  row subsampling) did not finish fitting within a 9-minute cell
  timeout - building many full-size, un-binned trees on ~2.16M rows
  turned out to be memory-bandwidth-bound, so parallel speedup across
  12 cores fell well short of what an initial timing calibration
  suggested. Switching to `max_samples=0.15` (each tree trains on a 15%
  bootstrap subsample) cut the fit to **91.4 seconds** for
  `n_estimators=60`, `max_depth=14`, `min_samples_leaf=10` - documented
  in full in section 4 above, including the re-measurement that
  justified it.
- **Overall: random forest is the best model of the five tried** - MAE
  27.34 / RMSE 54.13, vs. gradient-boosting's MAE 28.57 / RMSE 54.53,
  LightGBM's MAE 28.56 / RMSE 54.44, MLP's MAE 28.57 / RMSE 55.57, and
  the seasonal-naive baseline's MAE 39.34 / RMSE 77.25. Random forest
  beats gradient-boosting by ~4.3% MAE and ~0.7% RMSE - a small margin
  in RMSE but a clear one in MAE, larger than the near-ties among
  GBM/LightGBM/MLP seen in notebooks 08/09/11.
- **Per-station: random forest improves on the baseline for 22 of 23
  stations** (only `300037936` regresses, -4.2%) - the most consistent
  record of any model tried. Its per-station MAE is usually a few points
  higher than GBM/LightGBM's on stations where those two already do
  well, but the gap flips sharply in random forest's favor on the two
  previously-flagged stations (see below) - the improvement is
  concentrated exactly where the other models struggled most, rather
  than uniform.
- **The two flagged stations are fixed, not just improved**: unlike
  every other model class tried (GBM, LightGBM, Prophet, MLP - all of
  which regressed relative to the baseline on both), random forest
  *beats* the baseline on both `300037405` (+14.8% vs. baseline, vs.
  the other models' -8% to -16%) and `300038855` (+6.9% vs. baseline,
  vs. the other models' roughly -100% to -117% - more than double the
  baseline's error). See section 7 for the full write-up and a
  tentative explanation (bagging/subsample-averaging as an implicit
  regularizer against overfitting a stale historical relationship, most
  plausible for `300038855`'s genuine regime shift) that would benefit
  from further, more targeted diagnostics before being treated as
  settled.
- **Feature importance** (native `feature_importances_`, impurity-based,
  summed back from one-hot columns to original features): `total_count`
  overwhelmingly dominates (0.80 of total importance), far more
  concentrated than GBM's permutation-based ranking (0.36) or LightGBM's
  split-based ranking - consistent with a shallower forest
  (`max_depth=14`) relying more heavily on the single strongest signal
  and less on secondary interactions like `day_of_week`/`hour`
  (0.07/0.03 here vs. much larger shares in the other models).
  `day_of_week`, `hour`, and the lag features follow, broadly agreeing
  with the other notebooks on *which* features matter even though the
  relative magnitudes differ by importance-metric definition.
- **Conclusion**: for this problem, random forest is not just
  competitive with the boosted-tree models tried so far - it is the
  best overall performer among all five models, and the first to
  actually recover (not just narrow) the performance gap on both
  previously-flagged stations. This is a genuinely useful result:
  `CLAUDE.md`'s tech-stack list had flagged random forest as untried but
  plausible, and it turned out to matter specifically for the
  regime-shift station that every other model class struggled with.
  Caveats worth carrying forward: this result rests on a specific,
  empirically-tuned hyperparameter configuration (particularly
  `max_samples=0.15`) rather than sklearn defaults, so it is not simply
  "random forest wins" in general - it's this regularized, subsampled
  configuration that wins, and the regime-shift-station explanation
  above is a plausible hypothesis rather than a confirmed mechanism. A
  natural next step, if this holds up, would be to check whether
  applying similar subsampling/regularization ideas to the boosted-tree
  models (e.g. LightGBM's own `bagging_fraction`) recovers similar
  gains there, which would help distinguish "random forest specifically
  helps" from "subsample averaging in general helps" on this data.